# Matplotlib

**Domain:** Data Analysis & Research  ·  **from study list**  ·  **runnable:** yes

A refresher on Matplotlib — Python's foundational plotting library and the engine
underneath most of the scientific-Python visualization stack.

## 1. What & Why

**Matplotlib** is the foundational 2D (and basic 3D) plotting library for Python.
It turns arrays of numbers into pixels: line plots, scatter, bars, histograms,
heatmaps, contours, and arbitrarily customized composites. It dates to 2003 and
deliberately mimicked MATLAB's plotting API, which is why so much of it feels
imperative and stateful.

**The problem it solves.** You have NumPy arrays / Pandas columns and need to *see*
them — distributions, trends, correlations, residuals — both for quick exploration
and for publication-quality figures. Matplotlib gives you pixel-level control over
every element (ticks, spines, fonts, DPI, color) so the same code path works for a
throwaway `plt.plot(x, y)` in a REPL and a 600-DPI figure in a paper.

**When to reach for it.**
- You need full control over a figure's appearance, or a non-standard composite layout.
- You're building *on top of* it: Pandas `.plot()`, Seaborn, and scikit-learn's
  display helpers all render through Matplotlib, so knowing it lets you fix their output.
- You need a static raster/vector export (PNG/PDF/SVG) for a report, paper, or slide.

**When not to.** For fast statistical charts with sane defaults, reach for **Seaborn**
(built on Matplotlib). For interactive, zoomable, web-embedded charts, reach for
**Plotly**/**Bokeh**/**Altair**. Matplotlib's interactivity is limited and its defaults
require tuning — it's the assembly language of Python plotting, not the high-level DSL.

## 2. Mental Model

Think of a figure as a set of **nested containers**, like a picture frame holding canvases:

```
Figure  (the whole image / the page)
└── Axes  (one plot region: data area + its x/y axis, title, legend)
    ├── Axis  (the x-axis and y-axis objects: ticks, labels, scale)
    └── Artists  (everything drawn: Line2D, Patch, Text, Image, ...)
```

**`Figure` ≠ `Axes`.** A *Figure* is the entire image (it can hold many subplots).
An *Axes* is a single plot — the rectangular region with data, a title, and two
`Axis` objects. Almost everything you "draw" returns an **Artist** (a `Line2D`, a
`Rectangle`, a `Text`) that you can grab and mutate later.

**The two APIs — and why it matters:**

- **Implicit / pyplot (`plt.plot`, `plt.title`)** — MATLAB-style. There's a hidden
  "current figure / current axes"; each `plt.*` call acts on it. Fine for quick,
  single-plot scripts.
- **Explicit / object-oriented (`fig, ax = plt.subplots(); ax.plot(...)`)** — you hold
  the `Figure` and `Axes` objects and call methods on them. **Prefer this.** It is
  unambiguous with multiple subplots, composes into functions, and is what the docs
  now recommend.

The mental shortcut: `plt.plot(...)` is just `plt.gca().plot(...)` — "get current
axes, then plot." Once you internalize that, the two APIs stop feeling like two
libraries.

## 3. Key Concepts

- **Figure** — the top-level container; the whole image. `plt.figure()` or the `fig`
  from `plt.subplots()`. Owns `figsize` (inches) and `dpi` (the inches→pixels factor).
- **Axes** — one plot. The workhorse object: `ax.plot`, `ax.scatter`, `ax.set_xlabel`,
  `ax.legend`, `ax.set_xlim`. Not to be confused with **Axis** (singular), the x/y line
  with its ticks and scale.
- **`plt.subplots(nrows, ncols)`** — the canonical entry point. Returns
  `(fig, ax)` for a single plot or `(fig, axes_array)` for a grid.
- **Artist** — any drawable object. Plotting methods *return* artists
  (`line, = ax.plot(...)`), which you can restyle, animate, or remove.
- **pyplot vs OO API** — the stateful `plt.*` convenience layer vs explicit
  `fig`/`ax` method calls. See the Mental Model.
- **Backend** — the renderer. Interactive (`%matplotlib inline`, `widget`, `qtagg`) vs
  non-interactive/file (`Agg` for PNG, `pdf`, `svg`). Set with `matplotlib.use("Agg")`
  *before* importing pyplot for headless/server rendering.
- **`plt.show()` vs `fig.savefig()`** — display interactively vs write to a file. Use
  `bbox_inches="tight"` to crop whitespace and `dpi=` to control export resolution.
- **rcParams / styles** — global defaults (`plt.rcParams[...]`, `plt.style.use(...)`).
  One `plt.style.use("ggplot")` restyles every subsequent plot.
- **Colormaps & normalization** — map scalar values to colors (`viridis` is the
  perceptually-uniform default; avoid `jet`). A `Normalize` controls the value→color range.

## 4. Setup

Matplotlib is pure-Python + a small C extension; it installs cleanly with pip and
pulls in NumPy. In a notebook, `%matplotlib inline` (the Jupyter default) renders
figures as static images right under the cell.

In [1]:
# %pip install matplotlib numpy
import matplotlib
import numpy as np

# Use a non-interactive backend so this notebook executes headless (CI, nbconvert).
# In an interactive Jupyter session you'd normally skip this and rely on `inline`.
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("matplotlib", matplotlib.__version__)
print("numpy", np.__version__)
print("backend:", matplotlib.get_backend())

matplotlib 3.11.0
numpy 2.5.0
backend: Agg


## 5. Worked Examples

### Example 1 — The object-oriented API: a 2×2 grid of plot types

This is the pattern to memorize: `fig, axes = plt.subplots(...)`, then drive each
`ax` explicitly. It scales from one plot to a dashboard without changing style.

In [2]:
rng = np.random.default_rng(0)
x = np.linspace(0, 2 * np.pi, 200)

fig, axes = plt.subplots(2, 2, figsize=(9, 6), constrained_layout=True)
(ax_line, ax_scatter), (ax_hist, ax_bar) = axes

# 1. Line plot with two series, a legend, and labeled axes.
ax_line.plot(x, np.sin(x), label="sin")
ax_line.plot(x, np.cos(x), "--", label="cos")
ax_line.set_title("Line plot")
ax_line.set_xlabel("x")
ax_line.set_ylabel("amplitude")
ax_line.legend()

# 2. Scatter with a colormap mapped to a third variable.
xs = rng.normal(size=150)
ys = xs * 0.8 + rng.normal(scale=0.6, size=150)
sc = ax_scatter.scatter(xs, ys, c=xs * ys, cmap="viridis", s=20)
ax_scatter.set_title("Scatter (color = x*y)")
fig.colorbar(sc, ax=ax_scatter, shrink=0.8)

# 3. Histogram of a normal sample.
ax_hist.hist(rng.normal(size=2000), bins=30, color="steelblue", edgecolor="white")
ax_hist.set_title("Histogram")

# 4. Bar chart.
cats = ["A", "B", "C", "D"]
ax_bar.bar(cats, [5, 9, 3, 7], color=["#4c72b0", "#dd8452", "#55a868", "#c44e52"])
ax_bar.set_title("Bar chart")

fig.suptitle("One Figure, four Axes", fontsize=14)

# Save instead of show() because we're on the Agg backend.
fig.savefig("/tmp/mpl_grid.png", dpi=100, bbox_inches="tight")
print("saved figure:", fig.get_size_inches(), "inches @", fig.dpi, "dpi")
print("axes objects:", [type(a).__name__ for a in axes.ravel()])

saved figure: [9. 6.] inches @ 100.0 dpi
axes objects: ['Axes', 'Axes', 'Axes', 'Axes']


### Example 2 — Grab the returned Artist and restyle after the fact

Plotting methods *return* the artists they create. Capturing them lets you tweak
style, read data back, or animate — without re-plotting. Here we also show styling,
annotation, and twin axes.

In [3]:
x = np.linspace(0, 10, 100)
y = np.exp(-x / 3) * np.cos(2 * x)

with plt.style.context("ggplot"):
    fig, ax = plt.subplots(figsize=(8, 4))

    # ax.plot returns a list of Line2D; unpack the single line.
    (line,) = ax.plot(x, y, color="navy", linewidth=1.5, label="damped cosine")

    # Mutate the artist after creation — no re-plot needed.
    line.set_linestyle("--")
    line.set_alpha(0.8)

    # Annotate the global maximum.
    i_max = int(np.argmax(y))
    ax.annotate(
        f"max ≈ {y[i_max]:.2f}",
        xy=(x[i_max], y[i_max]),
        xytext=(x[i_max] + 1.5, y[i_max] + 0.2),
        arrowprops=dict(arrowstyle="->", color="black"),
    )

    # Twin y-axis sharing the same x — different scale on the right.
    ax2 = ax.twinx()
    ax2.plot(x, np.cumsum(np.abs(y)), color="darkorange", label="cumulative |y|")
    ax2.set_ylabel("cumulative |y|", color="darkorange")

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Captured artist + annotation + twin axis")
    ax.legend(loc="upper right")

    fig.savefig("/tmp/mpl_damped.png", dpi=100, bbox_inches="tight")

print("line color:", line.get_color(), "| style:", line.get_linestyle())
print("first 3 y-values read back from the artist:", line.get_ydata()[:3])

line color: navy | style: --
first 3 y-values read back from the artist: [1.         0.94722706 0.85960098]


## 6. Gotchas & Pitfalls

- **Figures leak memory in loops.** Every `plt.subplots()` / `plt.figure()` keeps a
  reference in pyplot's registry. In a loop that makes hundreds of figures (e.g.
  batch-saving plots), call `plt.close(fig)` after saving or you'll exhaust RAM and
  trip the "More than 20 figures opened" warning.
- **`plt.show()` blocks and clears.** In a script, `show()` blocks until you close the
  window and may clear the current figure — call `savefig()` **before** `show()`, not
  after, or you'll save a blank image.
- **pyplot's hidden state bites with multiple plots.** Interleaving `plt.*` calls when
  several figures exist draws onto whichever is "current," which is rarely what you
  meant. Use the OO API (`ax.plot`) once you have more than one Axes.
- **Backend must be set before importing pyplot.** `matplotlib.use("Agg")` after
  `import matplotlib.pyplot` is silently ignored. Set it first for headless rendering.
- **Overlapping labels / clipped titles.** The default `tight_layout` isn't automatic.
  Pass `constrained_layout=True` to `subplots()` (or call `fig.tight_layout()`), and
  use `bbox_inches="tight"` in `savefig` to stop labels getting cropped.
- **`jet` and rainbow colormaps mislead.** They aren't perceptually uniform and are
  bad for colorblind readers. Default to `viridis`/`cividis`/`magma`.
- **Strings vs numbers on an axis.** `ax.bar(["1","2","10"], ...)` treats labels as
  categories in insertion order, not sorted numbers. Convert to numeric first if you
  want numeric ordering/spacing.
- **DPI vs figsize confusion.** `figsize` is in *inches*; pixels = `figsize × dpi`.
  To make a bigger raster, raise `dpi`; to change proportions, change `figsize`.

## 7. When to Use vs Alternatives

| Tool | Best for | Trade-off vs Matplotlib |
|------|----------|--------------------------|
| **Matplotlib** | Full control, custom/composite static figures, publication output, as the engine under other libs | Verbose; defaults need tuning; weak interactivity |
| **Seaborn** | Fast statistical charts (distributions, regressions, categorical) with great defaults | Higher-level but *is* Matplotlib underneath — drop down to `ax` for fine control |
| **Pandas `.plot()`** | One-liner exploratory plots straight off a DataFrame | Thin wrapper over Matplotlib; limited customization |
| **Plotly / Bokeh / Altair** | Interactive, zoomable, web-embeddable charts; dashboards | Heavier, JS-based; harder to get pixel-perfect static export |
| **plotnine / ggplot2** | Grammar-of-graphics declarative style (R refugees) | Different mental model; smaller ecosystem in Python |

**Rule of thumb:** start in Seaborn or `df.plot()` for speed; drop to raw Matplotlib
when you need a custom layout or pixel-level polish; switch to Plotly/Altair when the
deliverable is interactive or web-embedded. Because Seaborn and Pandas return
Matplotlib `Axes`, knowing Matplotlib means you can *always* fix their output — that's
why it's worth keeping fresh. See also the [`seaborn`](seaborn.ipynb) and
[`plotly`](plotly.ipynb) notebooks.

## 8. Resources

- **Official docs** — https://matplotlib.org/stable/index.html
- **Quick-start / pyplot tutorial** — https://matplotlib.org/stable/tutorials/pyplot.html
- **The lifecycle of a plot (OO vs pyplot, explained well)** — https://matplotlib.org/stable/tutorials/lifecycle.html
- **Cheat sheets (printable, dense, excellent)** — https://matplotlib.org/cheatsheets/
- **Example gallery (copy-paste starting points)** — https://matplotlib.org/stable/gallery/index.html
- **Why `viridis` — a talk on colormaps** — https://bids.github.io/colormap/